# FASTA2GO - validação da etapa de extração

Fluxo integrado em lotes: ler o FASTA -> calcular as features -> consultar a UniProt ->
consolidar em CSV -> inserir o lote no DuckDB -> próximo lote.

**Como executar.** Abra este notebook no VS Code com o kernel do `.venv` do projeto. A Parte A
usa um banco descartável construído a partir de `tests/data/sample.fasta` (4 proteínas, incluindo
registros defeituosos) e chama a API real do UniProt. A Parte B inspeciona o banco produzido
pelo CLI com o Swiss-Prot real (a extração sempre começa de um banco vazio):

```bash
uv run python main.py run --recreate --limit 2000     # cria db/fasta2go.duckdb
```

As células usam `assert`: se algo estiver errado, a execução para.


## Preparação


In [ ]:
from pathlib import Path

import duckdb

from fasta2go import db, features, pipeline

SAMPLE = Path("tests/data/sample.fasta")
SCRATCH = Path("db/validation.duckdb")
REAL = Path("db/fasta2go.duckdb")
SCRATCH.unlink(missing_ok=True)      # a Parte A sempre começa de um banco vazio

conn = db.connect(SCRATCH)
db.init_schema(conn)


def show(sql: str, params=()) -> None:
    """Imprime o resultado de uma consulta como tabela simples."""
    result = conn.execute(sql, list(params))
    print(" | ".join(column[0] for column in result.description))
    for row in result.fetchall():
        print(" | ".join("" if value is None else str(value) for value in row))


## Extração integrada em lotes

Cada lote percorre o fluxo completo antes de o próximo começar: os registros são lidos e
validados, as features são calculadas, os accessions do lote são consultados na UniProt e as
sete tabelas do lote são gravadas em **uma transação**. Registros impossíveis de armazenar são
rejeitados e registrados em `ingestion_issues`; resíduos ambíguos ficam na sequência, mas são
sinalizados.


In [ ]:
counts = pipeline.extract(conn, SAMPLE, batch_size=2)
print(counts)

assert counts["inserted"] == 4        # quatro proteínas válidas e únicas
assert counts["rejected"] == 2        # sequência vazia e cabeçalho sem identificador
assert counts["duplicates"] == 1      # P69905 aparece duas vezes
assert counts["batches"] == 2         # quatro proteínas, duas por lote
assert counts["computed"] == 4
assert counts["queried"] == 3         # a proteína sem accession não é consultada
assert counts["matched"] + counts["not_found"] == 3

show("SELECT fasta_id, uniprot_accession, sequence_length FROM proteins ORDER BY fasta_id")
print()
show("SELECT line_number, fasta_id, issue, detail FROM ingestion_issues ORDER BY line_number")


## Features da sequência

Comprimento canônico, composição de aminoácidos (AAC), composição de dipeptídeos (DPC) e
propriedades físico-químicas. O denominador é sempre o número de resíduos canônicos: `X`, `B`,
`Z`, `U`, `O` e `*` são excluídos e contados em `noncanonical_residues`.


In [ ]:
show(
    """
    SELECT fasta_id, sequence_length, canonical_length, noncanonical_residues,
           round(molecular_weight, 1) AS mw, round(isoelectric_point, 2) AS pi,
           round(gravy, 3) AS gravy, round(sequence_entropy, 3) AS entropy
    FROM protein_features ORDER BY fasta_id
    """
)

# Os denominadores são verificáveis: toda linha de composição soma 100.
for table, columns in (
    ("amino_acid_composition", features.AAC_COLUMNS),
    ("dipeptide_composition", features.DPC_COLUMNS),
):
    expression = " + ".join(f'"{column}"' for column in columns)
    outside = conn.execute(
        f"SELECT count(*) FROM {table} WHERE abs(({expression}) - 100) > 1e-6"
    ).fetchone()[0]
    assert outside == 0, table
print("AAC e DPC: todas as linhas somam 100%")
print("colunas de DPC:", len(features.DPC_COLUMNS))


## Anotações de Gene Ontology

As anotações vêm de `uniProtKBCrossReferences` (database `GO`): identificador, aspecto (prefixo
`P`/`C`/`F` do `GoTerm`), nome, código de evidência e quem a atribuiu (`GoEvidenceType`), além
da proveniência (`evidences`, quando existe). Nenhum ancestral é propagado nesta etapa, e um
accession que a fonte não devolveu fica `not_found`, sem termo algum.


In [ ]:
show(
    """
    SELECT aspect, count(*) AS annotations, count(DISTINCT go_id) AS terms,
           count(*) FILTER (WHERE provenance IS NOT NULL) AS with_provenance
    FROM protein_go_annotations GROUP BY aspect ORDER BY aspect
    """
)
print()
show(
    """
    SELECT go_id, aspect, go_name, evidence_code, evidence_source, provenance
    FROM protein_go_annotations WHERE fasta_id = 'sp|P69905|HBA_HUMAN'
    ORDER BY aspect, go_id LIMIT 8
    """
)
print()
show(
    "SELECT fasta_id, status, uniprot_id, source_length "
    "FROM uniprot_annotations ORDER BY fasta_id"
)


## Consistência

As tabelas de features precisam ter exatamente as mesmas proteínas, a anotação precisa apontar
para o accession da própria proteína e nenhuma anotação GO pode pertencer a um accession que a
fonte não devolveu.


In [ ]:
counts = conn.execute(
    """
    SELECT
        (SELECT count(*) FROM proteins) AS proteins,
        (SELECT count(*) FROM sequence_features) AS features,
        (SELECT count(*) FROM amino_acid_composition) AS aac,
        (SELECT count(*) FROM dipeptide_composition) AS dpc,
        (SELECT count(*) FROM uniprot_annotations) AS annotated,
        (SELECT count(*) FROM protein_go_annotations) AS go_rows
    """
).fetchone()
print(counts)
assert counts[0] == counts[1] == counts[2] == counts[3] == 4
assert counts[4] == 3                              # as três proteínas com accession

with_accession = conn.execute(
    "SELECT count(*) FROM proteins WHERE uniprot_accession IS NOT NULL"
).fetchone()[0]
assert with_accession == counts[4]

# A anotação é sempre do accession da própria proteína...
assert conn.execute(
    """
    SELECT count(*) FROM uniprot_annotations a JOIN proteins p USING (source_file, fasta_id)
    WHERE a.uniprot_accession <> p.uniprot_accession
    """
).fetchone()[0] == 0

# ...e um accession ausente na fonte não recebe nenhum termo GO.
assert conn.execute(
    """
    SELECT count(*) FROM protein_go_annotations g
    LEFT JOIN uniprot_annotations a USING (source_file, fasta_id)
    WHERE a.status = 'not_found'
    """
).fetchone()[0] == 0

summary = pipeline.database_summary(conn)
for key, value in summary.items():
    print(f"{key:>22}: {value}")
assert summary["proteins"] == summary["with_features"] == summary["with_composition"]
assert summary["orphans"] == 0


## Parte B - Swiss-Prot real (somente leitura)

Banco construído com `uv run python main.py run --recreate --limit 2000`. Esta célula não altera
nada.


In [ ]:
if REAL.exists():
    real = duckdb.connect(str(REAL), read_only=True)
    counts = real.execute(
        "SELECT (SELECT count(*) FROM proteins) AS proteins, "
        "(SELECT count(*) FROM protein_go_annotations) AS go_rows, "
        "(SELECT count(DISTINCT (source_file || fasta_id)) FROM protein_go_annotations) "
        "AS proteins_with_go"
    ).fetchone()
    print("proteínas, linhas GO, proteínas com GO:", counts)

    print()
    print("termos de função molecular mais frequentes nesta amostra:")
    for row in real.execute(
        """
        SELECT go_id, any_value(go_name) AS term, count(*) AS proteins
        FROM protein_go_annotations WHERE aspect = 'molecular_function'
        GROUP BY go_id ORDER BY proteins DESC LIMIT 5
        """
    ).fetchall():
        print("  ", row)
    real.close()
else:
    print("db/fasta2go.duckdb não existe: rode `uv run main.py run --recreate --limit 2000`.")

conn.close()
